# UNETR BraTS2020: Training & Inference Pipeline

### ⚠️ Hướng dẫn dùng folder Shared:
1. Share thư mục `BraTS_Project` cho tài khoản Colab
2. Vào **"Shared with me"**
3. **"Add shortcut"** → **"My Drive"**

---

## 1. Setup Environment & Data

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

BASE_DRIVE_PATH = '/content/drive/MyDrive/BraTS_Project'
PROJECT_ZIP_PATH = os.path.join(BASE_DRIVE_PATH, 'colab_project.zip')
PROJECT_FOLDER_PATH = os.path.join(BASE_DRIVE_PATH, 'colab_project')
DATA_ZIP_PATH = os.path.join(BASE_DRIVE_PATH, 'brats2020_processed.zip')

In [ ]:
# 2. Copy Project Code
if os.path.exists('/content/project'):
    shutil.rmtree('/content/project')
os.makedirs('/content/project', exist_ok=True)

if os.path.exists(PROJECT_ZIP_PATH):
    print(f"Unzipping code...")
    !unzip -q "{PROJECT_ZIP_PATH}" -d /content/project
    if os.path.exists('/content/project/colab_project'):
        !mv /content/project/colab_project/* /content/project/
        !rmdir /content/project/colab_project
elif os.path.exists(PROJECT_FOLDER_PATH):
    os.rmdir('/content/project')
    shutil.copytree(PROJECT_FOLDER_PATH, '/content/project')

if os.path.exists('/content/project/trainers'):
    os.chdir('/content/project')
    print("✅ Project loaded")
else:
    print("❌ Setup failed")

In [ ]:
# 3. Install Requirements
!pip install -r requirements.txt
!pip install medpy

In [ ]:
# 4. Extract Data
if os.path.exists(DATA_ZIP_PATH):
    print(f"Unzipping data...")
    !unzip -q "{DATA_ZIP_PATH}" -d /content/project
    if os.path.exists("processed") and not os.path.exists("data/processed"):
        os.makedirs("data", exist_ok=True)
        shutil.move("processed", "data/processed")
    if os.path.exists("data/processed/3d/labeled"):
        print("✅ Data extracted")
    else:
        print("❌ Data extraction failed")

In [ ]:
# 5. Fix PyTorch 2.6 + Logger
print("🔧 Applying compatibility fixes...")

train_script = '/content/project/trainers/train_unetr3d_brats2020.py'

with open(train_script, 'r') as f:
    content = f.read()

# Fix 1: PyTorch weights_only
content = content.replace(
    'ckpt = torch.load(ckpt_path, map_location=device)',
    'ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)'
)

# Fix 2: Logger preserve logs
old_logger = '''    def __init__(self, path: str):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        self.data = []'''

new_logger = '''    def __init__(self, path: str):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        
        # Load existing data if file exists (for resume training)
        if os.path.exists(path):
            try:
                with open(path, "rb") as fp:
                    self.data = pickle.load(fp)
                print(f"[Logger] Loaded {len(self.data)} existing entries from {os.path.basename(path)}")
            except Exception as e:
                print(f"[Logger] Could not load existing log, starting fresh: {e}")
                self.data = []
        else:
            self.data = []'''

content = content.replace(old_logger, new_logger)

with open(train_script, 'w') as f:
    f.write(content)

print("✅ PyTorch fix applied (weights_only=False)")
print("✅ Logger fix applied (preserve existing logs)")

## 2. Training Configuration

In [ ]:
# @title Chế độ huấn luyện { run: "auto" }
TRAINING_MODE = "New Training" # @param ["New Training", "Resume Training"]
print(f"✅ Chế độ: {TRAINING_MODE}")

In [ ]:
# @title Cấu hình Resume Training { run: "auto" }
CHECKPOINT_PATH = "" # @param {type:"string"}
WANDB_RUN_ID = "" # @param {type:"string"}

if TRAINING_MODE == "Resume Training":
    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"✅ Checkpoint: {CHECKPOINT_PATH}")
    if WANDB_RUN_ID:
        print(f"✅ WandB ID: {WANDB_RUN_ID}")

In [ ]:
# @title Extend Training { run: "auto" }
EXTEND_TRAINING = False # @param {type:"boolean"}
NEW_MAX_EPOCH = 200 # @param {type:"integer"}

if TRAINING_MODE == "Resume Training" and EXTEND_TRAINING:
    print(f"✅ Extend to epoch {NEW_MAX_EPOCH}")

## 3. Training UNETR

In [ ]:
import sys
DRIVE_CHECKPOINT_PATH = os.path.join(BASE_DRIVE_PATH, 'checkpoints', 'Unetr_3d_checkpoint')

if TRAINING_MODE == "Resume Training":
    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"🔄 Resume Training")
        print(f"   Checkpoint: {CHECKPOINT_PATH}")
        if WANDB_RUN_ID:
            print(f"   WandB ID: {WANDB_RUN_ID}")
        if EXTEND_TRAINING:
            print(f"   Extend to: {NEW_MAX_EPOCH}")
        
        print("\n🚀 Starting resume training...\n")
        
        sys.path.insert(0, '/content/project')
        from trainers import train_unetr3d_brats2020
        
        train_unetr3d_brats2020.CFG['RESUME_CKPT'] = CHECKPOINT_PATH
        
        if WANDB_RUN_ID:
            train_unetr3d_brats2020.CFG['WANDB']['resume_id'] = WANDB_RUN_ID
        
        if EXTEND_TRAINING:
            train_unetr3d_brats2020.CFG['OPTIM']['MAX_EPOCH'] = NEW_MAX_EPOCH
            print(f"   ✅ MAX_EPOCH: 100 → {NEW_MAX_EPOCH}")
        
        import argparse
        original_argv = sys.argv.copy()
        sys.argv = ['train_unetr3d_brats2020.py', '--drive_path', DRIVE_CHECKPOINT_PATH]
        
        try:
            train_unetr3d_brats2020.main()
        finally:
            sys.argv = original_argv
    else:
        print("❌ Invalid checkpoint")
        raise ValueError("Invalid checkpoint path")
else:
    print("🆕 New Training\n")
    !python trainers/train_unetr3d_brats2020.py --drive_path "{DRIVE_CHECKPOINT_PATH}"

## 4. Inference & Evaluation

In [ ]:
# Chạy Inference UNETR với auto-save to Drive
DRIVE_INFERENCE_PATH = os.path.join(BASE_DRIVE_PATH, 'inference_results')
!python inference/inference_unetr_brats3d_fullvolume.py --drive_path "{DRIVE_INFERENCE_PATH}"

## 5. Visualization (Optional)

In [ ]:
# @title Cấu hình Visualization { run: "auto" }

# Case ID để visualize (bệnh nhân trong test set)
VIS_CASE_ID = "Brain_011" # @param {type:"string"}

# Số lát cắt (slices) để vẽ
VIS_NUM_SLICES = 1 # @param {type:"integer"}

# Lát cắt cố định (để trống nếu muốn auto-select)
VIS_FIXED_SLICES = "75" # @param {type:"string"}

# Modality nền (flair, t1, t1ce, t2)
VIS_BASE_MODALITY = "flair" # @param ["flair", "t1", "t1ce", "t2"]

print(f"✅ Sẽ visualize case: {VIS_CASE_ID}")
print(f"   Modality: {VIS_BASE_MODALITY}")
print(f"   Slices: {VIS_NUM_SLICES if not VIS_FIXED_SLICES else VIS_FIXED_SLICES}")

In [ ]:
# Chạy Visualization với auto-save to Drive
import sys

# Cập nhật config trong script
vis_script = '/content/project/scripts/visualize_unetr_results.py'
with open(vis_script, 'r') as f:
    vis_content = f.read()

# Update CASE_ID
vis_content = vis_content.replace(
    '"CASE_ID": "Brain_091"',
    f'"CASE_ID": "{VIS_CASE_ID}"'
)

# Update BASE_MODALITY
vis_content = vis_content.replace(
    '"BASE_MODALITY": "flair"',
    f'"BASE_MODALITY": "{VIS_BASE_MODALITY}"'
)

# Update NUM_SLICES
vis_content = vis_content.replace(
    '"NUM_SLICES": 6',
    f'"NUM_SLICES": {VIS_NUM_SLICES}'
)

# Update FIXED_SLICES if provided
if VIS_FIXED_SLICES:
    fixed_list = [int(x.strip()) for x in VIS_FIXED_SLICES.split(',') if x.strip()]
    vis_content = vis_content.replace(
        '"FIXED_SLICES": [88]',
        f'"FIXED_SLICES": {fixed_list}'
    )

with open(vis_script, 'w') as f:
    f.write(vis_content)

# Run visualization
DRIVE_VIS_PATH = os.path.join(BASE_DRIVE_PATH, 'visualization_results')
!python scripts/visualize_unetr_results.py --drive_path "{DRIVE_VIS_PATH}"

## 6. Backup Results

In [ ]:
# Backup experiments (optional - most results already saved to Drive)
!zip -r experiments_backup.zip experiments/
BACKUP_PATH = os.path.join(BASE_DRIVE_PATH, 'experiments_result.zip')
!cp experiments_backup.zip "{BACKUP_PATH}"
print(f"✅ Backup completed!")